<a href="https://colab.research.google.com/github/marketcalls/Age-Gender-Emotion-Analyzer/blob/main/Gemma3_270M_Financial_NewsDataset_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + "0.0.32.post2" if v == "2.8.0" else "0.0.29.post3"
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

In [2]:
# ===============================
# STEP 1: LOAD GEMMA 3 270M MODEL
# ===============================

from unsloth import FastModel
import torch

print("Loading Gemma 3 270M model...")

max_seq_length = 2048  # Good for financial sentences

# Load the exact model from your working code
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-270m-it",  # Instruction-tuned version
    max_seq_length = max_seq_length,
    load_in_4bit = False,   # Keep as False for 270M model
    load_in_8bit = False,   # Keep as False for 270M model
    full_finetuning = False, # Use LoRA for efficiency
    # token = "hf_...", # Add if using gated models
)

print(f"Model loaded: unsloth/gemma-3-270m-it")
print(f"Max sequence length: {max_seq_length}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Gemma 3 270M model...
==((====))==  Unsloth 2025.8.9: Fast Gemma3 patching. Transformers: 4.55.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Model loaded: unsloth/gemma-3-270m-it
Max sequence length: 2048


In [3]:
# ===============================
# STEP 2: SETUP LORA
# ===============================

print("Setting up LoRA fine-tuning...")

# Use exact same LoRA config as your working code
model = FastModel.get_peft_model(
    model,
    r = 128, # Same as your config
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128,      # Same as your config
    lora_dropout = 0,      # Same as your config
    bias = "none",         # Same as your config
    use_gradient_checkpointing = "unsloth", # Same optimization
    random_state = 3407,   # Same random state
    use_rslora = False,    # Same as your config
    loftq_config = None,   # Same as your config
)

print("LoRA configuration complete")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Setting up LoRA fine-tuning...
Unsloth: Making `model.base_model.model.model` require gradients
LoRA configuration complete
Trainable parameters: 30,375,936


In [4]:
# ===============================
# STEP 3: SETUP CHAT TEMPLATE (SAME AS YOUR CODE)
# ===============================

from unsloth.chat_templates import get_chat_template

print("Setting up Gemma 3 chat template...")

# Use exact same chat template setup
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma3",  # Same as your code
)

print("Chat template configured")

Setting up Gemma 3 chat template...
Chat template configured


In [26]:
# ===============================
# STEP 4: LOAD COMBINED FINANCIAL TWEETS DATASET (38K)
# ===============================

from datasets import load_dataset
import pandas as pd

print("Loading Combined Financial Tweets dataset (38K examples)...")

# Load the much larger Combined Financial Tweets dataset
financial_dataset = load_dataset("TimKoornstra/financial-tweets-sentiment")
print(f"Loaded {len(financial_dataset['train'])} financial tweets")


# Show sample
print("Sample entry:")
print(financial_dataset['train'][0])

Loading Combined Financial Tweets dataset (38K examples)...
Loaded 38091 financial tweets
Sample entry:
{'tweet': '$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT', 'sentiment': 2, 'url': 'https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment'}


In [27]:
financial_dataset['train'][1]

{'tweet': '$CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3',
 'sentiment': 2,
 'url': 'https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment'}

In [28]:
# ===============================
# STEP 5: CONVERT TO CHATML FORMAT (ADAPTED FROM YOUR CODE)
# ===============================

def convert_financial_to_chatml(example):
    """Convert Combined Financial Tweets to ChatML format like your chess example"""

    # Map labels to sentiment (this dataset uses 0=neutral, 1=positive, 2=negative)
    label_map = {0: "neutral", 1: "positive", 2: "negative"}
    sentiment = label_map[example['sentiment']]

    # Create system message for financial sentiment analysis
    system_message = "You are a financial sentiment analysis expert. Analyze financial tweets and social media posts to classify sentiment as positive, negative, or neutral with detailed reasoning."

    # Create user input
    user_message = f"Analyze the sentiment of this financial tweet: {example['tweet']}"

    # Create detailed assistant response (similar to your chess format)
    assistant_response = f"""SENTIMENT: {sentiment.upper()}

ANALYSIS: This financial tweet expresses {sentiment} sentiment toward the market or specific assets. {'The language indicates optimistic market outlook, positive developments, or bullish sentiment that could drive buying interest.' if sentiment == 'positive' else 'The language suggests pessimistic market views, concerning developments, or bearish sentiment that could drive selling pressure.' if sentiment == 'negative' else 'The content presents balanced or factual information without strong directional bias toward market movements.'}

CONFIDENCE: High (multi-source aggregated data)

MARKET_IMPLICATION: This social media sentiment suggests {'bullish' if sentiment == 'positive' else 'bearish' if sentiment == 'negative' else 'neutral'} retail investor positioning and could indicate broader market sentiment trends relevant for short-term trading decisions."""

    return {
        "conversations": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_response}
        ]
    }

print("Converting dataset to ChatML format...")

# Apply the conversion (same pattern as your chess code)
dataset = financial_dataset['train'].map(convert_financial_to_chatml)

print("Dataset converted to conversation format")
print("Sample conversation:")
print(dataset[0])

# Optional: Take a subset for faster training if needed
# dataset = dataset.select(range(10000))  # Use first 10K for faster training
# print(f"Using subset of {len(dataset)} examples for training")

Converting dataset to ChatML format...


Map:   0%|          | 0/38091 [00:00<?, ? examples/s]

Dataset converted to conversation format
Sample conversation:
{'tweet': '$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT', 'sentiment': 2, 'url': 'https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment', 'conversations': [{'content': 'You are a financial sentiment analysis expert. Analyze financial tweets and social media posts to classify sentiment as positive, negative, or neutral with detailed reasoning.', 'role': 'system'}, {'content': 'Analyze the sentiment of this financial tweet: $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT', 'role': 'user'}, {'content': 'SENTIMENT: NEGATIVE\n\nANALYSIS: This financial tweet expresses negative sentiment toward the market or specific assets. The language suggests pessimistic market views, concerning developments, or bearish sentiment that could drive selling pressure.\n\nCONFIDENCE: High (multi-source aggregated data)\n\nMARKET_IMPLICATION: This social media sentime

In [29]:
# ===============================
# STEP 6: APPLY FORMATTING (SAME AS YOUR CODE)
# ===============================

def formatting_prompts_func(examples):
    """Same formatting function as your code"""
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts, }

print("Applying chat template formatting...")

dataset = dataset.map(formatting_prompts_func, batched = True)

print("Formatting complete")
print("Sample formatted text:")
print(dataset[100]['text'][:500] + "...")

Applying chat template formatting...


Map:   0%|          | 0/38091 [00:00<?, ? examples/s]

Formatting complete
Sample formatted text:
<start_of_turn>user
You are a financial sentiment analysis expert. Analyze financial tweets and social media posts to classify sentiment as positive, negative, or neutral with detailed reasoning.

Analyze the sentiment of this financial tweet: O'Reilly Automotive stock price target cut to $405 from $415 at J.P. Morgan<end_of_turn>
<start_of_turn>model
SENTIMENT: NEGATIVE

ANALYSIS: This financial tweet expresses negative sentiment toward the market or specific assets. The language suggests pessi...


In [30]:
# ===============================
# STEP 7: TRAINING SETUP
# ===============================

from trl import SFTTrainer, SFTConfig

print("Setting up training configuration...")

# Use exact same training config as your working code
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Same as your config
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,    # Same as your config
        gradient_accumulation_steps = 1,    # Same as your config
        warmup_steps = 5,                   # Same as your config
        max_steps = 500,                    # Increased for larger dataset
        learning_rate = 3e-5,               # Slightly lower LR for larger dataset
        logging_steps = 1,                  # Same as your config
        optim = "adamw_8bit",              # Same as your config
        weight_decay = 0.01,               # Same as your config
        lr_scheduler_type = "linear",       # Same as your config
        seed = 3407,                       # Same as your config
        output_dir="financial_tweets_outputs",  # Updated output directory
        report_to = "none",                # Same as your config
    ),
)

print("Training configuration complete")

Setting up training configuration...
Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/38091 [00:00<?, ? examples/s]

Training configuration complete


In [31]:
# ===============================
# STEP 8: TRAIN ON RESPONSES ONLY
# ===============================

from unsloth.chat_templates import train_on_responses_only

print("Setting up response-only training...")

# Use exact same approach as your code
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",      # Same as your config
    response_part = "<start_of_turn>model\n",        # Same as your config
)

print("Response-only training configured")

# Verify masking works (same as your code)
print("Verifying training masking...")
print("Input IDs sample:")
print(tokenizer.decode(trainer.train_dataset[100]["input_ids"])[:200] + "...")

print("\nLabels (masked) sample:")
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")[:200] + "...")


Setting up response-only training...


Map (num_proc=2):   0%|          | 0/38091 [00:00<?, ? examples/s]

Response-only training configured
Verifying training masking...
Input IDs sample:
<bos><start_of_turn>user
You are a financial sentiment analysis expert. Analyze financial tweets and social media posts to classify sentiment as positive, negative, or neutral with detailed reasoning....

Labels (masked) sample:
                                                                       SENTIMENT: NEGATIVE

ANALYSIS: This financial tweet expresses negative sentiment toward the market or specific assets. The langua...


In [32]:
# ===============================
# STEP 9: SHOW MEMORY STATS
# ===============================

# Show memory stats
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
    print(f"{start_gpu_memory} GB of memory reserved.")
else:
    print("Running on CPU")
    start_gpu_memory = 0
    max_memory = 0

GPU = Tesla T4. Max memory = 14.741 GB.
2.027 GB of memory reserved.


In [33]:
# ===============================
# STEP 10: TRAIN THE MODEL
# ===============================

print("Starting training...")
print("Expected time: 5-10 minutes on GPU, 15-30 minutes on CPU")

# Train the model (same call as your code)
trainer_stats = trainer.train()

print("Training completed!")

Starting training...
Expected time: 5-10 minutes on GPU, 15-30 minutes on CPU


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 38,091 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 30,375,936 of 298,474,112 (10.18% trained)


Step,Training Loss
1,7.725000
2,7.576800
3,7.359500
4,6.216500
5,5.138200
6,3.891100
7,2.889600
8,2.114800
9,1.665200
10,1.330600


Unsloth: Will smartly offload gradients to save VRAM!
Training completed!


In [34]:
# ===============================
# STEP 11: SHOW FINAL STATS
# ===============================

if torch.cuda.is_available():
    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
    used_percentage = round(used_memory / max_memory * 100, 3)
    lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

    print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
    print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
    print(f"Peak reserved memory = {used_memory} GB.")
    print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
    print(f"Peak reserved memory % of max memory = {used_percentage} %.")
    print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

print(f"Final training loss: {trainer_stats.training_loss:.4f}")

232.9177 seconds used for training.
3.88 minutes used for training.
Peak reserved memory = 2.982 GB.
Peak reserved memory for training = 0.955 GB.
Peak reserved memory % of max memory = 20.229 %.
Peak reserved memory for training % of max memory = 6.479 %.
Final training loss: 0.1142


In [35]:
# ===============================
# STEP 12: INFERENCE TESTING
# ===============================

print("\nTesting the fine-tuned model...")

# Test on real news from RedboxGlobal India tweets
test_sentences = [
    "RELIANCE INDUSTRIES: CO ACQUIRES REMAINING 6.1% STAKE IN NAUYAAN SHIPYARD, MAKING IT A WHOLLY OWNED STEP-DOWN SUBSIDIARY FOR ₹45.32 CRORE",
    "NIFTY 50 INDEX DOWN BY 20.78% DUE TO GLOBAL CRISIS",
    "TIKTOK WEBSITE RESUMES ACCESS IN INDIA AFTER 5 YEARS, BUT APP REMAINS BANNED ON PLAY STORE & APP STORE",
    "MAX HEALTHCARE AND INTERGLOBE AVIATION TO BE INCLUDED IN NIFTY 50 || INDUSIND BANK AND HERO MOTOCORP TO BE EXCLUDED FROM NIFTY 50 || CHANGES IN NIFTY 50 SHALL BECOME EFFECTIVE FROM SEPTEMBER 30",
    "MUTHOOT FINANCE: FITCH UPGRADES CO'S LONG-TERM RATING TO 'BB+'; OUTLOOK STABLE",
    "OSWAL GREENTECH: CO ACQUIRES 4.99% STAKE (66.99 LAKH SHARES) IN OSWAL AGRO MILLS FOR ₹50.91 CR VIA OPEN MARKET, RAISING PROMOTER HOLDING",
    "STAR CEMENT: STAR CEMENT NORTH EAST DECLARED PREFERRED BIDDER FOR LIMESTONE BLOCK",
    "FOSECO INDIA: CO SIGNS DEFINITIVE AGREEMENT TO ACQUIRE A 75% STAKE IN MORGANITE CRUCIBLE INDIA LIMITED, TO STRENGTHEN ITS FOUNDRY BUSINESS IN INDIA"
]

for i, sentence in enumerate(test_sentences):
    print(f"\nTest {i+1}: {sentence}")

    # Create messages in same format as your code
    messages = [
        {'role': 'system', 'content': 'You are a financial sentiment analysis expert using market-based sentiment labels. Analyze financial headlines and classify sentiment based on actual market impact, not subjective opinion.'},
        {"role": 'user', 'content': f"Analyze the market sentiment of this financial headline: {sentence}"}
    ]

    # Apply chat template (same as your code)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True,
    ).removeprefix('<bos>')

    # Generate response (same settings as your code)
    from transformers import TextStreamer
    print("AI Analysis:")
    _ = model.generate(
        **tokenizer(text, return_tensors = "pt").to("cuda" if torch.cuda.is_available() else "cpu"),
        max_new_tokens = 125,
        temperature = 1,     # Same as your settings
        top_p = 0.95,        # Same as your settings
        top_k = 64,          # Same as your settings
        streamer = TextStreamer(tokenizer, skip_prompt = True),
    )
    print("-" * 80)


Testing the fine-tuned model...

Test 1: RELIANCE INDUSTRIES: CO ACQUIRES REMAINING 6.1% STAKE IN NAUYAAN SHIPYARD, MAKING IT A WHOLLY OWNED STEP-DOWN SUBSIDIARY FOR ₹45.32 CRORE
AI Analysis:
SENTIMENT: NEUTRAL

ANALYSIS: This financial headline expresses neutral sentiment toward the market or specific assets. The content presents balanced or factual information without strong directional bias toward market movements.

CONFIDENCE: High (multi-source aggregated data)

MARKET Sentiment: NEUTRAL

CONFIDENCE: High (market-data derived labels)

VISUAL QU gantGGGE: NEGATIVE
DIGITAL QU text: This social media sentiment suggests neutral consumer buying intent toward this market-trepided asset.<end_of_turn>
--------------------------------------------------------------------------------

Test 2: NIFTY 50 INDEX DOWN BY 20.78% DUE TO GLOBAL CRISIS
AI Analysis:
SENTIMENT: NEGATIVE

ANALYSIS: This financial headline expresses negative sentiment toward the market or specific assets. The market show

In [37]:
# ===============================
# STEP 13: SAVE THE MODEL (SAME OPTIONS AS YOUR CODE)
# ===============================

print("\nSaving the fine-tuned model...")

# Save LoRA adapters (same as your code)
model.save_pretrained("financial_tweets_lora_model")
tokenizer.save_pretrained("financial_tweets_lora_model")

print("LoRA adapters saved to 'financial_tweets_lora_model'")

# Optional: Save to different formats (same as your code)
print("\nAdditional saving options:")
print("1. Set save_merged_16bit = True to save merged 16-bit model")
print("2. Set save_merged_4bit = True to save merged 4-bit model")
print("3. Set save_gguf = True to save GGUF format for llama.cpp")
print("4. Set push_to_hub = True to upload to Hugging Face")

# Configuration flags
save_merged_16bit = False
save_merged_4bit = False
save_gguf = False
push_to_hub = False

# Save to float16 merged model
if save_merged_16bit:
    print("Saving merged 16-bit model...")
    model.save_pretrained_merged("financial_model_16bit", tokenizer, save_method = "merged_16bit")
    print("16-bit model saved")

# Save to 4bit merged model
if save_merged_4bit:
    print("Saving merged 4-bit model...")
    model.save_pretrained_merged("financial_model_4bit", tokenizer, save_method = "merged_4bit")
    print("4-bit model saved")

# Save to GGUF format
if save_gguf:
    print("Saving GGUF model...")
    model.save_pretrained_gguf(
        "financial_gemma_gguf",
        quantization_type = "Q8_0",
    )
    print("GGUF model saved")

# Push to Hugging Face Hub
if push_to_hub:
    hf_token = "hf_..."  # Add your token here
    repo_name = "your_username/gemma-3-270m-financial-sentiment"

    print(f"Pushing to Hugging Face: {repo_name}")
    model.push_to_hub(repo_name, token = hf_token)
    tokenizer.push_to_hub(repo_name, token = hf_token)
    print("Model uploaded to Hugging Face")

print("\nFinancial tweets sentiment analysis model training complete!")
print("=" * 60)
print("Results Summary:")
print(f"Model: Gemma 3 270M fine-tuned on Combined Financial Tweets")
print(f"Training examples: {len(dataset)}")
print(f"Training time: {round(trainer_stats.metrics.get('train_runtime', 0)/60, 2)} minutes")
print(f"Final loss: {trainer_stats.training_loss:.4f}")
print(f"Memory usage: ~0.5GB RAM")
print(f"Task: Social media financial sentiment analysis")
print("\nUsage:")
print("1. Use 'financial_tweets_lora_model' for LoRA inference")
print("2. Test with financial tweets and social media posts")
print("3. Integrate into social sentiment trading strategies")
print("4. Deploy for real-time social media sentiment analysis")


Saving the fine-tuned model...
LoRA adapters saved to 'financial_tweets_lora_model'

Additional saving options:
1. Set save_merged_16bit = True to save merged 16-bit model
2. Set save_merged_4bit = True to save merged 4-bit model
3. Set save_gguf = True to save GGUF format for llama.cpp
4. Set push_to_hub = True to upload to Hugging Face

Financial tweets sentiment analysis model training complete!
Results Summary:
Model: Gemma 3 270M fine-tuned on Combined Financial Tweets
Training examples: 38091
Training time: 3.88 minutes
Final loss: 0.1142
Memory usage: ~0.5GB RAM
Task: Social media financial sentiment analysis

Usage:
1. Use 'financial_tweets_lora_model' for LoRA inference
2. Test with financial tweets and social media posts
3. Integrate into social sentiment trading strategies
4. Deploy for real-time social media sentiment analysis


In [38]:
# ===============================
# STEP 14: LOADING SAVED MODEL (OPTIONAL)
# ===============================

def load_financial_model():
    """Load the saved financial sentiment model"""
    print("Loading saved financial sentiment model...")

    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "financial_lora_model",
        max_seq_length = 2048,
        load_in_4bit = False,
    )

    print("Financial sentiment model loaded!")
    return model, tokenizer

# Uncomment to test loading:
# model, tokenizer = load_financial_model()

print("\nResources:")
print("• Unsloth Documentation: https://docs.unsloth.ai/")
print("• Discord Support: https://discord.gg/unsloth")
print("• GitHub: https://github.com/unslothai/unsloth")
print("• Twitter News Sentiment: https://dl.acm.org/doi/10.1145/3637528.3671629")


Resources:
• Unsloth Documentation: https://docs.unsloth.ai/
• Discord Support: https://discord.gg/unsloth
• GitHub: https://github.com/unslothai/unsloth
• Twitter News Sentiment: https://dl.acm.org/doi/10.1145/3637528.3671629
